# Nemotron — preparación del replay físico de una hora

Este notebook **no ejecuta inferencia** ni modifica los resultados offline/streaming acelerados. Usa el dataset y la evaluación ya congelada para construir un audio continuo, una selección reproducible y un cue-sheet con referencias humanas.

In [ ]:
!pip install -q soundfile

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/Nacholazabal/subtitle_overlay_fw.git'
REPO_BRANCH = 'dev/nemotron'
REPO_DIR = Path('/content/subtitle_overlay_fw')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH], check=True)
required = REPO_DIR / 'server/evaluation/prepare_physical.py'
if not required.is_file():
    raise RuntimeError('La branch de GitHub todavía no contiene el preparador físico. Primero pusheá estos cambios.')
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('Repo:', REPO_DIR, 'branch:', REPO_BRANCH)

In [ ]:
from server.evaluation.dataset_manifest import verify_archive, safe_extract_archive, discover_dataset_root

DRIVE_ROOT = Path('/content/drive/MyDrive/TESIS')
ARCHIVE = DRIVE_ROOT / 'stt_benchmarks/mediaspeech_es/v1.1/ES.tgz'
EVALUATION_DIR = DRIVE_ROOT / 'stt_evaluations/mediaspeech_es/mediaspeech-es-v1.1__nemotron-560-600-2__v1'
BUNDLE_DIR = DRIVE_ROOT / 'stt_evaluations/mediaspeech_es/mediaspeech-es-v1.1__nemotron-560-600-2__physical-v1'
EXTRACT_ROOT = Path('/content/mediaspeech-es-v1.1')

print(verify_archive(ARCHIVE))
if not list(EXTRACT_ROOT.rglob('*.flac')):
    safe_extract_archive(ARCHIVE, EXTRACT_ROOT)
DATASET_ROOT = discover_dataset_root(EXTRACT_ROOT)
print('Dataset:', DATASET_ROOT)
print('Frozen evaluation:', EVALUATION_DIR)
print('Physical bundle:', BUNDLE_DIR)

In [ ]:
from server.evaluation.prepare_physical import build_bundle

if (BUNDLE_DIR / 'bundle.json').is_file():
    print('El bundle ya existe; no se sobrescribe:', BUNDLE_DIR)
else:
    result = build_bundle(DATASET_ROOT, EVALUATION_DIR, BUNDLE_DIR)
    print('Clips:', result['selection']['selected_clips'])
    print('Minutos de habla:', result['selection']['selected_speech_sec'] / 60)
    print('Minutos totales:', result['audio']['duration_sec'] / 60)

In [ ]:
ZIP_PATH = BUNDLE_DIR.parent / f'{BUNDLE_DIR.name}.zip'
if not ZIP_PATH.is_file():
    shutil.make_archive(str(BUNDLE_DIR), 'zip', root_dir=BUNDLE_DIR.parent, base_dir=BUNDLE_DIR.name)
print('Listo:', ZIP_PATH)
print('Descargá/descomprimí ese ZIP en Windows. Después WSL ejecutará el replay físico apuntando a esa carpeta.')